In [1]:
import numpy as np
import pandas as pd
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from ax.utils.stats.model_fit_stats import MSE
from botorch.models import SingleTaskGP
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import plotly.express as px

from gpytorch.kernels import MaternKernel
from ax.models.torch.botorch_modular.kernels import DefaultRBFKernel, ScaleMaternKernel
from gpytorch.kernels.linear_kernel import LinearKernel
from gpytorch.kernels.rbf_kernel import RBFKernel

from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

# Data Carpentry

In [2]:
GrdSrch_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-A1_PGCI-GrdSrch-[27]-P3O1/raw-data_2023-03-10_PtA1-PGCI-GrdSrch-[27]-P3O1_Stykke-4.csv")
RndSrch_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-A2_PGCI-RndSrch-[27]-P3O1/raw-data_2023-03-10_PtA2-PGCI-RndSrch-[27]-P3O1_Stykke-4.csv")
GrdSrch_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)
RndSrch_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)
BOpt_8SP_3It_df = pd.read_csv("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/Project-1_BO4BT/ExperimentalSeries-1_PGXOpt/Part-C_HypOptComp_PGCI/Part-B5_PGCI-BOpt-[8,3,3,3,3,3,3,1]-P3O1/raw-data_2023-03-20_PtB5-PGCI-BOpt-[8,3,3,3,3,3,3,1]-P3O1-Stykke-4.csv")
BOpt_8SP_3It_df.rename(columns={"x1": "s1", "x2": "s2", "x3": "b1", "delta_polymer_mass_pct": "DeltaPolymerMass_pct", "polymer_start_mass_g": "StartPolymerMass_g", "polymer_end_mass_pct": "EndPolymerMass_pct"},inplace=True)

In [3]:
df = pd.concat(objs=[GrdSrch_df,RndSrch_df,BOpt_8SP_3It_df]) # Model mk16 using nu = 0.5 and obtaining RMSE = 1.148
df.drop(columns=["mould_position","G_stoichiometry","CA_stoichiometry","IA_stoichiometry","StartPolymerMass_g","EndPolymerMass_pct"],inplace=True)
df['DeltaPolymerMass_pct']=df['DeltaPolymerMass_pct']*-1
X = df[["s1","s2","b1"]].to_numpy()
y = df["DeltaPolymerMass_pct"].to_numpy()

In [4]:
trials = 10
rmse_vals = []
r1 = 0
r2 = 10000
SeedRange = list(range(r1,r2))
seeds = []

for i in range(trials):
    seed = np.random.choice(SeedRange)
    seeds.append(seed)
    # Get 27 testing samples from only the grid and pseudorandom exercise
    X_train, X_test, y_train, y_test = train_test_split(X[0:54],y[0:54],test_size=0.5,random_state=seed)

    # Append all the rest of the samples to the training set
    y_train = np.append(y_train,y[54::])
    X_train = np.concatenate((X_train,X[54::]), axis=0)

    # Set up the GP to be trained
    client = Client()
    parameters = [  RangeParameterConfig(name="s1", parameter_type="float", bounds=(0, 1)),
                    RangeParameterConfig(name="s2", parameter_type="float", bounds=(0, 1)),
                    RangeParameterConfig(name="b1", parameter_type="float", bounds=(0, 1)),
    ]
    client.configure_experiment(parameters=parameters)

    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            ModelConfig(
                botorch_model_class=SingleTaskGP,

                # covar_module_class=MaternKernel,
                # covar_module_options={"nu": 2.5},

                # covar_module_class=MaternKernel,
                # covar_module_options={"nu": 1.5},

                covar_module_class=MaternKernel,
                covar_module_options={"nu": 0.5},

                # covar_module_class=RBFKernel,
            ),
        ],
        eval_criterion=MSE,
        allow_batched_models=False,
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            "acquisition_options": {},
        },
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1"
    objective = f"{metric_name}"

    client.configure_optimization(objective=objective)

    for array,target in zip(X_train,y_train):
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": target})

    client.get_next_trials(max_trials=1)

    # Validate the GP trained
    y_pred = []
    for i,j in zip(X_test,y_test):
        y_pred.append(client.predict([{"s1":i[0],"s2":i[1],"b1":i[2]}])[0]["t1"][0])
    y_pred = np.array(y_pred)
    rmse_vals.append(root_mean_squared_error(y_test, y_pred))

print(np.average(rmse_vals))

1.0024194486561808


In [5]:
# Finding the best performing seed
print(np.min(rmse_vals))
print(seeds[np.argmin(rmse_vals)])

0.8257376362036445
7589


In [ ]:
# nu = 0.5      seed 7589 gives rmse of 0.826
# nu = 1.5      seed 2206 gives rmse of 0.756
# nu = 2.5      seed 4500 gives rmse of 0.656